In [610]:
import pickle
import sys
import copy
import time

import cobra

import multiprocessing
import multiprocessing.pool
# from multiprocessing import Process
# from threading import Thread

sys.path.insert(1, '/home/hratch/Projects/human_me/scripts/')
from utils import functions as func
from utils import parameters as params

In [274]:
# import pandas as pd
# build_files_path = '/data2/hratch/human_me/build_files/'
# human_model = cobra.io.load_json_model('/data2/hratch/human_me/input_files/toy_model.json')
# full_model = cobra.io.load_json_model('/data2/hratch/human_me/input_files/recon2_2.json')
# full_model_public = cobra.io.read_sbml_model(lp_path + 'recon2_2.xml')

# required_metabolites = pd.read_csv(build_files_path + 'required_metabolic_model_metabolites.csv', index_col = 0)

# me_model_og = copy.deepcopy(me_model)

In [611]:
from tqdm import tqdm

In [612]:
#For some reason, this protein complex used to catalyze the formation of the pre40s complex causes infeasiblity. 
#Including any of the 2 of the complex subcomponents in the precursors_fail list makes it work. Or their earlier 
#versions (up to unfolded_protein_c).

error_metabolites = ['pre40s_rrna_protein_COMPLEX_FORMATIONn_protein_complex[n]']
precurors_work = ['HGNC:21173_folded_protein[n]', 'HGNC:32790_folded_protein[n]']
precursors_fail = ['HGNC:25542_folded_protein[n]', 'HGNC:29100_folded_protein[n]']

# folded_protein[n] <-- folded_protein[c] <-- unfolded_protein[c] <-- 
# adding unfolded_protein[c] of precursors fail works too, don't need both of the precursors fail, just 1...
error_metabolites = precursors_fail.copy()

In [616]:
lp_path = '/data2/hratch/human_me/test_lp/'

def remove_metabolite(test_metabolites = [], mu_val = 0.01):
    
    with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
        me_model = pickle.load(handle)
    
    me_metabolites = [m.id for m in me_model.metabolites if 'deg_proxy' in m.id or ('mrna[n]' in m.id and 'premrna' not in m.id and 'lariats' not in m.id)]
    me_metabolites += error_metabolites

    for tm in test_metabolites:
        me_metabolites.remove(tm)
    
    ra = []
    for mm_id in me_metabolites: #me_metabolites:
        try:
            mm_obj = me_model.metabolites.get_by_id(mm_id)
        except:
            mm_obj = params.human_model.metabolites.get_by_id(mm_id)
        r = cobra.Reaction('TEST_' + mm_obj.id)
        r.add_metabolites({mm_obj: 1}, reversibly = True)
        ra.append(r)
    if len(ra) > 0:
        me_model.add_reactions(ra)
    sln, status, _ = me_model.solve_lp(mu_val = mu_val)
    return sln, status

In [ ]:
sln, status = remove_metabolite()

In [550]:
me_model.reactions.get_by_id('pre40s_rrna_protein_COMPLEX_FORMATIONn_COMPLEX_FORMATIONn').reaction

'HGNC:21173_folded_protein[n] + HGNC:25542_folded_protein[n] + HGNC:29100_folded_protein[n] + HGNC:32790_folded_protein[n] <=> pre40s_rrna_protein_COMPLEX_FORMATIONn_protein_complex[n]'

In [551]:
me_model.metabolites.get_by_id('HGNC:21173_folded_protein[n]').reactions

frozenset({<ME_Reaction HGNC:21173_IMPORTtn at 0x7f311f0c2748>,
           <ME_Reaction HGNC:21173_folded_protein[n]_DEUBIQUITINATIONn at 0x7f311f0c2b38>,
           <ME_Reaction HGNC:21173_folded_protein[n]_POLYUBIQUITINATIONn at 0x7f311f0c2908>,
           <Reaction pre40s_rrna_protein_COMPLEX_FORMATIONn_COMPLEX_FORMATIONn at 0x7f3177f71d30>})

In [552]:
sln[me_model.reactions.index('HGNC:21173_IMPORTtn')]

4.286012674106478e-17

In [553]:
sln[me_model.reactions.index('HGNC:21173_CYTOSOLIC_PROTEIN_FOLDING')]

4.286012674106478e-17

In [554]:
sln[me_model.reactions.index('HGNC:32790_IMPORTtn')]

4.286012674106478e-17

In [555]:
sln[me_model.reactions.index('HGNC:32790_IMPORTtn')]

4.286012674106478e-17

In [556]:
sln[me_model.reactions.index('HGNC:32790_CYTOSOLIC_PROTEIN_FOLDING')]

4.286012674106478e-17

In [557]:
sln[me_model.reactions.index('HGNC:25542_IMPORTtn')]

4.286012674106478e-17

In [558]:
sln[me_model.reactions.index('HGNC:25542_CYTOSOLIC_PROTEIN_FOLDING')]

4.286012674106478e-17

In [559]:
sln[me_model.reactions.index('HGNC:29100_IMPORTtn')]

4.286012674106478e-17

In [560]:
sln[me_model.reactions.index('HGNC:29100_CYTOSOLIC_PROTEIN_FOLDING')]

4.286012674106478e-17

In [561]:
sln[me_model.reactions.index('pre40s_rrna_protein_COMPLEX_FORMATIONn_COMPLEX_FORMATIONn')]

4.286012674106478e-17

In [562]:
sln[me_model.reactions.index('40s_MATURATION')]

7.10434122843673e-11

In [570]:
sln[me_model.reactions.index('RIBOSOME_COMPLEX_FORMATIONc')]

3.552170614218365e-11

In [605]:
lp_path = '/data2/hratch/human_me/test_lp/'
error_metabolites = ['pre40s_rrna_protein_COMPLEX_FORMATIONn_protein_complex[n]']

precurors_work = ['HGNC:21173_folded_protein[n]', 'HGNC:32790_folded_protein[n]']
precursors_fail = ['HGNC:25542_folded_protein[n]', 'HGNC:29100_folded_protein[n]']


precursors_fail = ['HGNC:25542_unfolded_protein[c]', 'HGNC:25542_unfolded_protein[c]']


error_metabolites = precursors_fail.copy()

def rm2(test_metabolites = [], mu_val = 0.01):
    
    with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
        me_model = pickle.load(handle)
    
    me_metabolites = [m.id for m in me_model.metabolites if 'deg_proxy' in m.id or ('mrna[n]' in m.id and 'premrna' not in m.id and 'lariats' not in m.id)]
    
#     me_metabolites = [m.id for m in me_model.metabolites if ('mrna[n]' in m.id and 'premrna' not in m.id and 'lariats' not in m.id)]
    
#     me_metabolites = [m.id for m in me_model.metabolites if 'deg_proxy' in m.id]


    me_metabolites += error_metabolites
    me_metabolites = sorted(set(me_metabolites))

    for tm in test_metabolites:
        me_metabolites.remove(tm)
    
    ra = []
    for mm_id in me_metabolites: #me_metabolites:
        try:
            mm_obj = me_model.metabolites.get_by_id(mm_id)
        except:
            mm_obj = params.human_model.metabolites.get_by_id(mm_id)
        r = cobra.Reaction('TEST_' + mm_obj.id)
        r.add_metabolites({mm_obj: 1}, reversibly = True)
        ra.append(r)
    if len(ra) > 0:
        me_model.add_reactions(ra)
    sln, status, _ = me_model.solve_lp(mu_val = mu_val)
    return sln, status

In [606]:
sln, status = rm2()

# removing deg_proxy fails, removing mrna[n] fails

Getting MINOS parameters...
Done in 141.163 seconds with status 0


In [591]:
me_model.reactions.get_by_id('HGNC:25542_CYTOSOLIC_PROTEIN_FOLDING').reaction

'4.31782671493333e5*mu  8.63565342986667e7 CYTOSOLIC_PROTEIN_FOLDING_protein_complex[c] + HGNC:25542_unfolded_protein[c] + 402 atp[c] + 402 h2o[c] --> HGNC:25542_folded_protein[c] + 402 adp[c] + 402 h[c] + 402 pi[c]'

In [600]:
me_model.reactions.get_by_id('HGNC:25542_TRANSLATION_ELONGATIONc').reaction

'mu/(mu + 0.02) HGNC:25542_mrna[c] + 0.0693147180559945/(mu + 0.02) HGNC:25542_mrna_deg_proxy + 2.94012994757356e6*mu  5.88025989514712e8 TRANSLATION_ELONGATIONc_complex_protein_complex[c] + 92.59811995999553 biomass_tRNA + 47 charged_generic_A_trna[c] + 7 charged_generic_C_trna[c] + 51 charged_generic_D_trna[c] + 66 charged_generic_E_trna[c] + 36 charged_generic_F_trna[c] + 52 charged_generic_G_trna[c] + 26 charged_generic_H_trna[c] + 23 charged_generic_I_trna[c] + 65 charged_generic_K_trna[c] + 79 charged_generic_L_trna[c] + 27 charged_generic_M_trna[c] + 18 charged_generic_N_trna[c] + 48 charged_generic_P_trna[c] + 42 charged_generic_Q_trna[c] + 50 charged_generic_R_trna[c] + 44 charged_generic_S_trna[c] + 37 charged_generic_T_trna[c] + 57 charged_generic_V_trna[c] + 10 charged_generic_W_trna[c] + 19 charged_generic_Y_trna[c] + 804 gtp[c] + 805 h2o[c] --> HGNC:25542_unfolded_protein[c] + 91.8089613000004 biomass_protein + 804 gdp[c] + 804 generic_trna[c] + 1608 h[c] + 804 pi[c]'

In [ ]:
# works with just HGNC:25542_folded_protein[c]

In [ ]:
# mrna[n] requirement: no flux through transcription elongation and transcription processing!!
# mrna_deg_proxy requirement: no flux through transcription degradation reaction
# additionally, no flux through protein degradation (polyub reaction, deubiquitination, or proteosome) reaction; 
# 2 lariat degradation reactions created